# UAE Mobile Intelligence - Zone Aggregation (Ookla + WorldPop -> H3 res 7)

Builds the two zone-level tables everything downstream depends on:

1. **`ookla_zones_uae.parquet`** -- one row per `(H3 cell, quarter)`, aggregated from the raw
   UAE-clipped tiles built by [`01_ookla_collection.ipynb`](01_ookla_collection.ipynb).
2. **`population_zones_uae.parquet`** -- one row per H3 cell, population summed from the WorldPop
   raster acquired by [`02_worldpop_collection.ipynb`](02_worldpop_collection.ipynb).

Uses H3 resolution **7**, chosen and justified empirically in
[`04_h3_resolution_choice.ipynb`](04_h3_resolution_choice.ipynb) (read back below rather than
hard-coded, so this notebook can't silently drift out of sync with that decision).

**Why aggregate each quarter separately before anything else:** individual Ookla tiles churn --
only ~11% of quadkeys persist across all 8 quarters (see
[`00_ookla_dataset_overview.ipynb`](00_ookla_dataset_overview.ipynb) §7). Grouping by
`(H3 cell, quarter)` here, rather than joining tiles across quarters first, is what avoids
throwing that data away: H3 cells are far more stable across quarters than raw tiles are.

In [1]:
import json
from pathlib import Path

import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import rasterio

In [2]:
RESOLUTION_DECISION = json.loads(Path("../data/processed/h3_resolution.json").read_text())
H3_RESOLUTION = RESOLUTION_DECISION["chosen_resolution"]

print("Using H3 resolution:", H3_RESOLUTION)
print("Decision source: 04_h3_resolution_choice.ipynb ->", RESOLUTION_DECISION["justification"][:120], "...")

Using H3 resolution: 7
Decision source: 04_h3_resolution_choice.ipynb -> Resolution 7 reproduces the brief's own cited density figures (median 4 tests/zone, ~83% of zones below 30 tests) on ind ...


## Part 1 -- Ookla tiles into zone-quarters

Assign each tile to its H3 cell (`(lat, lon)` = `(tile_y, tile_x)`, the same swap gotcha as
everywhere else in this dataset), then aggregate per `(h3_cell, quarter)` with test-weighted
averages -- a zone with 1 test shouldn't count as much as a zone with 100 in the same cell's
average.

In [3]:
tiles = gpd.read_parquet("../data/processed/ookla_tiles_uae.parquet")
tiles = pd.DataFrame(tiles.drop(columns=["geometry", "tile_geometry"]))

tiles["h3_cell"] = [
    h3.latlng_to_cell(lat, lon, H3_RESOLUTION)
    for lat, lon in zip(tiles["tile_y"], tiles["tile_x"])
]

print("Tile-quarter rows:", len(tiles))
print("Unique zones touched (any quarter):", tiles["h3_cell"].nunique())

Tile-quarter rows: 50860
Unique zones touched (any quarter): 3400


Loaded latency (`avg_lat_down_ms`) is the metric the brief asks for where populated (~99% of
UAE rows), so it's aggregated separately from unloaded latency (`avg_lat_ms`, always populated) --
each using its own test-count denominator, since a handful of tiles are missing the loaded figure.

In [4]:
tiles["w_download"] = tiles["avg_d_kbps"] * tiles["tests"]
tiles["w_upload"] = tiles["avg_u_kbps"] * tiles["tests"]
tiles["w_latency"] = tiles["avg_lat_ms"] * tiles["tests"]

has_loaded_lat = tiles["avg_lat_down_ms"].notna()
tiles["w_latency_loaded"] = tiles["avg_lat_down_ms"].fillna(0) * tiles["tests"]
tiles["tests_loaded_lat"] = tiles["tests"].where(has_loaded_lat, 0)

zones = (
    tiles.groupby(["h3_cell", "quarter"])
    .agg(
        n_tiles=("quadkey", "count"),
        tests=("tests", "sum"),
        devices=("devices", "sum"),
        w_download=("w_download", "sum"),
        w_upload=("w_upload", "sum"),
        w_latency=("w_latency", "sum"),
        w_latency_loaded=("w_latency_loaded", "sum"),
        tests_loaded_lat=("tests_loaded_lat", "sum"),
    )
    .reset_index()
)

zones["download_mbps"] = zones["w_download"] / zones["tests"] / 1000
zones["upload_mbps"] = zones["w_upload"] / zones["tests"] / 1000
zones["latency_ms"] = zones["w_latency"] / zones["tests"]
zones["latency_loaded_ms"] = zones["w_latency_loaded"] / zones["tests_loaded_lat"]

zones = zones.drop(columns=["w_download", "w_upload", "w_latency", "w_latency_loaded"])

print("Zone-quarter rows:", len(zones))
zones.head()

Zone-quarter rows: 13597


,h3_cell,quarter,n_tiles,tests,devices,tests_loaded_lat,download_mbps,upload_mbps,latency_ms,latency_loaded_ms
0,87438411effffff,2024Q3,1,1,1,1,42.041,17.201,23.0,214.0
1,87438411effffff,2024Q4,1,1,1,1,3.956,11.607,21.0,1802.0
2,874384508ffffff,2025Q2,1,1,1,1,31.127,3.563,25.0,281.0
3,874384508ffffff,2025Q3,1,4,2,4,117.812,5.593,31.0,1978.0
4,874384508ffffff,2025Q4,1,5,4,5,128.126,12.992,26.0,156.0


**Note on `devices`:** summed across tiles within a zone. Since the public dataset carries no
device identifiers, a device active in two tiles of the same zone would be counted twice -- this is
an upper-bound proxy for distinct devices, not an exact count. Worth restating when the Confidence
Score uses it later.

**Conservation check:** no measurements should be dropped by aggregation -- total `tests` in the
zone table must equal total `tests` in the source tile table, both overall and per quarter.

In [5]:
assert zones["tests"].sum() == tiles["tests"].sum(), "Test count not conserved by aggregation!"

per_quarter_check = (
    zones.groupby("quarter")["tests"].sum().sort_index()
    == tiles.groupby("quarter")["tests"].sum().sort_index()
).all()
assert per_quarter_check, "Per-quarter test counts don't match!"

print("Total tests, tile table:", tiles["tests"].sum())
print("Total tests, zone table:", zones["tests"].sum())
print("Per-quarter conservation OK:", per_quarter_check)

Total tests, tile table: 327174
Total tests, zone table: 327174
Per-quarter conservation OK: True


In [6]:
out_path = Path("../data/processed/ookla_zones_uae.parquet")
zones.to_parquet(out_path, index=False)
print(f"Saved: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")

Saved: ..\data\processed\ookla_zones_uae.parquet (413.1 KB)


## Part 2 -- WorldPop population into the same H3 zones

The brief wants population "conserved to the person" when aggregated. Rather than rasterizing each
H3 hexagon individually (slow, ~1,800+ separate raster reads), assign every **populated** raster
pixel's center coordinate directly to an H3 cell and sum -- since every populated pixel is counted
exactly once, the national total is conserved by construction, and it's a single vectorized pass
instead of thousands of small ones. Pixels with zero (or nodata) population are skipped since they
can't move the sum.

In [7]:
raster_path = Path("../data/raw/worldpop/are_pop_2026_CN_100m_R2025A_v1.tif")

with rasterio.open(raster_path) as src:
    band = src.read(1, masked=True)
    transform = src.transform

filled = band.filled(0)
pop_rows, pop_cols = np.where(filled > 0)
pop_values = filled[pop_rows, pop_cols]

pixel_lons, pixel_lats = rasterio.transform.xy(transform, pop_rows, pop_cols)
pixel_lons = np.asarray(pixel_lons)
pixel_lats = np.asarray(pixel_lats)

print("Populated pixels:", len(pop_values))
print("Raw raster national total (sanity check vs. 02_worldpop_collection.ipynb):", f"{pop_values.sum():,.0f}")

Populated pixels: 1127338
Raw raster national total (sanity check vs. 02_worldpop_collection.ipynb): 11,476,873


In [8]:
pixel_h3 = [
    h3.latlng_to_cell(lat, lon, H3_RESOLUTION)
    for lat, lon in zip(pixel_lats, pixel_lons)
]

pop_df = pd.DataFrame({"h3_cell": pixel_h3, "population": pop_values})
population_zones = pop_df.groupby("h3_cell", as_index=False)["population"].sum()

print("Populated H3 zones:", len(population_zones))
population_zones.head()

Populated H3 zones: 7880


,h3_cell,population
0,874384006ffffff,7.394281
1,874384012ffffff,0.517119
2,874384014ffffff,8.913122
3,874384016ffffff,15.085392
4,874384023ffffff,3.803755


**Conservation check:** the zone-level population total must equal the raster's own national
total (11,476,873, already validated in
[`02_worldpop_collection.ipynb`](02_worldpop_collection.ipynb)), since every populated pixel maps to
exactly one H3 cell.

In [9]:
raster_total = float(pop_values.sum())
zone_total = float(population_zones["population"].sum())

print(f"Raster total:  {raster_total:,.0f}")
print(f"Zone total:    {zone_total:,.0f}")
assert np.isclose(raster_total, zone_total), "Population not conserved by H3 aggregation!"
print("Conservation OK (exact -- every populated pixel maps to exactly one H3 cell).")

Raster total:  11,476,873
Zone total:    11,476,872
Conservation OK (exact -- every populated pixel maps to exactly one H3 cell).


In [10]:
out_path = Path("../data/processed/population_zones_uae.parquet")
population_zones.to_parquet(out_path, index=False)
print(f"Saved: {out_path} ({out_path.stat().st_size / 1024:.1f} KB)")

Saved: ..\data\processed\population_zones_uae.parquet (96.8 KB)


## Summary

- `data/processed/ookla_zones_uae.parquet` -- one row per `(H3 res-7 cell, quarter)`, with
  test-weighted download/upload/latency, summed tests/devices, and tile count per zone-quarter.
  Ready for the Experience Index and Confidence Score work (Phase 2).
- `data/processed/population_zones_uae.parquet` -- one row per H3 res-7 cell, population summed
  from WorldPop with the national total exactly conserved. Ready for population-exposure layers and
  the priority engine (Phase 3).
- Both conservation checks passed: no measurements or population were silently dropped.

Next: [`06_first_uae_map.ipynb`](06_first_uae_map.ipynb) joins these two tables for the latest
quarter and renders the first interactive UAE map, clearing the Phase 1 gate.